# 03 · Windows and physics features (first half)

## Purpose
Build and verify the shared preprocessing that every model will see, on a small training subset only:
whole-recording cleaning (microphone events and a fixed band-stop for the ~22 kHz line), windowing with per-window
shaft speed, and the physics features (R1). No full caches, no predictive models and no spectrograms here; the only
model is a diagnostic RandomForest that checks whether the band-stop removed a shortcut.

## Inputs
`metadata.csv`, `data/raw_npy/`, `configs/base.yaml`, and the caches written by `scripts/06_bandstop.py`
(band-stop measurements) and `scripts/07_features.py` (subset features), each named by the sha256 of what produced it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tools, the settings and <code>metadata.csv</code>, and finds the cached results of the band-stop script. If a cache is missing it stops and says which command to run.</div>

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import freqz, welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from ml.config import config_sha256, load_config
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, cache_file
from ml.preprocess import apply_fir_centred, bandstop_taps, propose_stopband
from ml.splits import split_hash

cfg = load_config(DEFAULT_CONFIG)
pre = cfg.preprocess
FS = pre.fs_hz
bs = pre.bandstop
FIG = SYSTEM_CODE_DIR / "results" / "figures"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")

caches = {
    "measure": cache_file("06_measure", pre.mic, ext="parquet"),
    "check": cache_file("06_check", pre.mic, ext="parquet"),
    "verify": cache_file("06_verify", pre.mic, pre.bandstop, ext="parquet"),
    "shortcut": cache_file("06_shortcut", pre.mic, pre.bandstop, ext="parquet"),
}
missing = [k for k, p in caches.items() if not p.exists()]
assert not missing, f"run: python scripts/06_bandstop.py {' '.join(missing)}"
print(f"preprocess config sha256 {config_sha256(pre)[:16]}…   split hash {split_hash(meta)[:12]}…")

## 1. The band-stop for the ~22 kHz line

**Exposure note.** Stage 1c (notebook 02) already measured this line across *all* classes, including faults, as an
exploratory check. That is how its folder-linked level was found. The stop band below is nevertheless **set from
`split_closed` train normal recordings only**; fault recordings are used afterwards only to *check* it, never to widen
it. Normal is never withheld, so this setting is safe for every LOCO fold and for the healthy-only track.

### Setting it
Per train normal recording and channel 1–7: the main line (strongest bin in 21.8–22.3 kHz, 0.2 Hz bins, interpolated)
and its ±60/±120 Hz sidebands, plus the line's drift across 1 s windows. The rule: cover the observed extent of the
lines, add a stated **25 Hz margin** each side (about the spread of the line across recordings), round outward to 5 Hz.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tone measurements of the 34 train normal recordings, summarises where the line and its sidebands sit, applies the stop-band rule, and checks the result equals the band stored in the settings.</div>

In [ ]:
tone_n = pd.read_parquet(caches["measure"])
present = tone_n[tone_n.snr_db > 20]
print(f"{tone_n.rec_id.nunique()} train normal recordings, {len(present)} of {len(tone_n)} recording × channel pairs "
      f"with the line > 20 dB above the local floor")
summary = pd.DataFrame({
    "frequency min (Hz)": [present.f0_hz.min()] + [present[f"f{o}_hz"].min() for o in ("-120", "-60", "+60", "+120")],
    "frequency max (Hz)": [present.f0_hz.max()] + [present[f"f{o}_hz"].max() for o in ("-120", "-60", "+60", "+120")],
}, index=["line", "−120 Hz", "−60 Hz", "+60 Hz", "+120 Hz"]).round(1)
print(f"drift of the line within a recording (1 s windows): median {present.drift_hz.median():.2f} Hz, "
      f"max {present.drift_hz.max():.2f} Hz")
band = propose_stopband(tone_n, present_db=20, margin_hz=25, round_hz=5)
print(band)
assert (band["stop_lo_hz"], band["stop_hi_hz"]) == (bs.stop_lo_hz, bs.stop_hi_hz), "configs/base.yaml bandstop is out of date"
summary

The filter: a symmetric, odd-length **linear-phase FIR** (`scipy.signal.firwin`, Kaiser window), applied by centred
convolution so its group delay is removed (zero-phase alignment). Design targets: 80 dB across the stop band and
200 Hz transitions each side, which `kaiserord` turns into 1,257 taps (12.6 ms half-length) and Kaiser β = 7.857.
Edges, tap count and window are stored in `bandstop` in the config.

### Checking it on other train and val recordings (no retuning)
The same measurement on every other train and val recording, including fault recordings. Anything outside the stop
band is reported, not used to widen it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the same tone measurements for all other train and val recordings and counts, for the line and each sideband, how many recording × channel pairs fall outside the stop band, and in which classes.</div>

In [ ]:
tone_c = pd.read_parquet(caches["check"])
pc = tone_c[tone_c.snr_db > 20]
rows = []
for name, fcol, scol, thr in [("line", "f0_hz", "snr_db", 20)] + [
        (f"{o} Hz", f"f{o}_hz", f"snr{o}_db", 10) for o in ("-120", "-60", "+60", "+120")]:
    s = pc[pc[scol] > thr]
    out = s[(s[fcol] < bs.stop_lo_hz) | (s[fcol] > bs.stop_hi_hz)]
    rows.append({"line": name, "present": len(s), "outside the stop band": len(out), "files": out.rec_id.nunique(),
                 "range (Hz)": f"{out[fcol].min():.0f}–{out[fcol].max():.0f}" if len(out) else "",
                 "classes (files)": out.groupby("class10").rec_id.nunique().to_dict() if len(out) else ""})
print(f"{tone_c.rec_id.nunique()} other train/val recordings; line frequency {pc.f0_hz.min():.1f}–{pc.f0_hz.max():.1f} Hz")
pd.DataFrame(rows)

**The line itself always falls inside the stop band**, but in some sessions it sits higher than in any train normal
recording (up to 22,156 Hz), so its **+120 Hz sideband lands in the upper transition band in 46 recordings** (up to
22,276 Hz; 26 of them horizontal misalignment) and the +60 Hz sideband in 5 (all horizontal misalignment). There the
attenuation is partial. Reported, not fixed: widening the band from fault recordings is not allowed, and any later
data-driven change must respect the relevant fold.

### (a) The designed frequency response

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Computes the filter's frequency response on a fine grid and reports the smallest attenuation inside the stop band, the largest ripple outside the transitions, and where the response crosses −0.01, −3 and −80 dB.</div>

In [ ]:
taps = bandstop_taps(pre)
w, h = freqz(taps, worN=2**20, fs=FS)
H = 20 * np.log10(np.abs(h) + 1e-15)
in_stop = (w >= bs.stop_lo_hz) & (w <= bs.stop_hi_hz)
passband = (w <= bs.stop_lo_hz - bs.transition_hz) | (w >= bs.stop_hi_hz + bs.transition_hz)


def crossing(level: float, lower: bool) -> float:
    side = (w > 21_000) & (w < bs.stop_lo_hz + 1) if lower else (w > bs.stop_hi_hz - 1) & (w < 23_000)
    ww, hh = w[side], H[side]
    return float(ww[np.argmax(hh < level)] if lower else ww[np.argmax(hh > level)])


resp = pd.Series({
    "taps": taps.size, "symmetric": bool(np.allclose(taps, taps[::-1])),
    "min stop-band attenuation (dB)": -H[in_stop].max(),
    "max pass-band ripple (dB)": np.abs(H[passband]).max(),
    **{f"lower edge {lvl} dB (Hz)": crossing(lvl, True) for lvl in (-0.01, -3, -80)},
    **{f"upper edge {lvl} dB (Hz)": crossing(lvl, False) for lvl in (-80, -3, -0.01)},
})
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
axes[0].plot(w / 1e3, H, lw=0.8)
axes[0].set(xlabel="frequency (kHz)", ylabel="gain (dB)", ylim=(-120, 5), title="Band-stop response, 0–25 kHz")
axes[1].plot(w, H, lw=0.8)
axes[1].axvspan(bs.stop_lo_hz, bs.stop_hi_hz, color="red", alpha=0.1, label="stop band")
axes[1].set(xlim=(21_500, 22_600), ylim=(-120, 5), xlabel="frequency (Hz)", title="zoom")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG / "03_bandstop_response.png")
resp.round(4)

### (b) Synthetic signals: a tone at the stop-band frequency plus broadband noise

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds 5 s test signals of white noise plus a tone at the typical line frequency, at the upper end seen in fault sessions, and at both stop-band edges; filters them; and reports how much each tone drops and how much the noise spectrum changes outside the stop band and transitions.</div>

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(5 * FS) / FS
half = (bs.numtaps - 1) // 2
rows = []
for f_tone in (22_028.0, 22_156.0, bs.stop_lo_hz, bs.stop_hi_hz):
    noise = 0.05 * rng.normal(size=t.size)
    tone_sig = 0.5 * np.sin(2 * np.pi * f_tone * t)
    y = apply_fir_centred(noise + tone_sig, taps)
    core = slice(half, t.size - half)
    probe = np.exp(-2j * np.pi * f_tone * t[core])
    a_in = 2 * np.abs(np.mean((noise + tone_sig)[core] * probe))
    a_out = 2 * np.abs(np.mean(y[core] * probe))
    fw, p_in = welch(noise[core], fs=FS, nperseg=8192)
    _, p_out = welch(apply_fir_centred(noise, taps)[core], fs=FS, nperseg=8192)
    keep = ((fw < bs.stop_lo_hz - bs.transition_hz) | (fw > bs.stop_hi_hz + bs.transition_hz)) & (fw > 20)
    rows.append({"tone (Hz)": f_tone, "tone attenuation (dB)": 20 * np.log10(a_in / a_out),
                 "noise change outside (max |dB|)": np.abs(10 * np.log10(p_out[keep] / p_in[keep])).max()})
pd.DataFrame(rows).round(3)

At the stop-band edges the tone drops by 80.9–81.0 dB, matching the 80 dB design; inside the band it drops further (82.7 dB at 22,028 Hz, 98.6 dB at 22,156 Hz). The noise spectrum outside the stop band and its transitions changes by at most 0.001 dB.

### (c) Real train recordings
Three train recordings per class (30), from the scan cache: the line's attenuation per recording (where it is present),
the largest change outside the stop band and its transitions (100 Hz bands), and the **measured** recording-edge
effect. The edge effect is measured by filtering a 2 s crop as if it were a recording and comparing it with the same
2 s cut from the filtered full recording (which had real signal on both sides).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the per-recording verification results and summarises them: minimum, median and maximum over the 30 train recordings, for the line attenuation, the change outside the stop band, and how many milliseconds from each recording edge the filter's output stays more than −40 dB and −60 dB away from the true result.</div>

In [ ]:
ver = pd.read_parquet(caches["verify"])
print(f"{len(ver)} train recordings; filter half-length {(bs.numtaps - 1) / 2 / FS * 1e3:.1f} ms")
ver.drop(columns=["rec_id", "class10"]).describe().T[["min", "50%", "max"]].round(4)

The edge effect is shorter than the filter's half-length: the output is within −40 dB of the true result after
4–7 ms and within −60 dB after 8–11 ms, at both ends. The first and last 1 s windows therefore contain at most ~11 ms of
slightly disturbed signal. Because the band-stop runs on the whole recording (rule 11), only those two recording edges
are affected, never the window boundaries inside a recording.

### Diagnostic shortcut test
Is the stop band still class-informative after filtering? Per-channel energy in the stop band (log) → class10,
RandomForest, 5-fold `cv_fold` CV on train + val, against a label-permutation baseline (10 permutations). As a further
diagnostic, the same test on two bands next to the stop band (21.3–21.6 and 22.45–22.75 kHz) that the filter does not
touch. The result is a finding to report, not a reason to retune the filter.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the per-recording stop-band energies from the scan cache, trains a random forest with 5-fold cross-validation on the energies before filtering, after filtering, and in the untouched neighbouring bands, and compares each macro-F1 with the score when the class labels are randomly shuffled.</div>

In [ ]:
sc = pd.read_parquet(caches["shortcut"])
chans = list(pre.channels[1:])
y = sc.class10.to_numpy()


def oof_f1(X: np.ndarray, labels: np.ndarray) -> float:
    pred = np.empty(len(sc), dtype=object)
    for f in range(5):
        tr, te = (sc.cv_fold != f).to_numpy(), (sc.cv_fold == f).to_numpy()
        pred[te] = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X[tr], labels[tr]).predict(X[te])
    return f1_score(labels, pred, average="macro")


feats = {k: np.log10(sc[[f"{k}_{c}" for c in chans]].to_numpy() + 1e-20)
         for k in ("before", "after", "before_adjacent")}
rng = np.random.default_rng(0)
perm = [oof_f1(feats["after"], rng.permutation(y)) for _ in range(10)]
shortcut_bs = pd.Series({
    "stop-band energy before filtering": oof_f1(feats["before"], y),
    "stop-band energy after filtering": oof_f1(feats["after"], y),
    "neighbouring bands (untouched)": oof_f1(feats["before_adjacent"], y),
    "permutation baseline, mean": np.mean(perm),
    "permutation baseline, max of 10": np.max(perm),
})
print(f"{len(sc)} train + val recordings, 10 classes (chance ≈ 0.10)")
shortcut_bs.round(3)

**Finding.** The stop band's energy predicts the class almost perfectly before filtering (macro-F1 0.98). After
filtering it still reaches 0.91, far above the permutation baseline (0.10). But the untouched neighbouring bands reach
0.94 on their own. What is left is therefore not the line: the broadband noise level near 22 kHz itself differs between
recording sessions and so between classes. The band-stop removes the line as intended; the session-dependent noise
floor is a separate shortcut risk for any representation that reaches these frequencies (Stage 2C).

## Findings (first half)
_To be completed at the end of the subset run._

## Outputs written
- `results/figures/03_bandstop_response.png`